# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammadfaheem8988/FlyRank-AI-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### 1. Method Choice and Rationale

For our binary classification lane (predicting whether an underperforming/decaying URL recovers organic clicks in the subsequent month), we evaluate two candidate models against our Week 4 heuristic baseline:

1. **Primary Model: Gradient Boosted Trees (LightGBM / HistGradientBoosting)**
   * **Why**: Organic search logs have heavy right-skewed distributions, zero-inflated clicks, and multi-variable interaction effects. In Week 4, we observed that high impressions + position < 1.0 does not always equal high yield due to SERP feature cannibalization. Gradient boosted decision trees model non-linear interactions automatically without manual scaling or brittle nested if-statements.
2. **Reference Benchmark: Regularized Logistic Regression**
   * **Why**: Serves as a transparent linear baseline to confirm that gains from tree ensembles are genuine signal learning rather than memorization of high-cardinality noise.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### 2. Validation & Split Design

* **Split Strategy: Client-Grouped Train/Test Split (`GroupShuffleSplit` on `client_hash_id`)**
* **Leakage Defense**: URLs belonging to the same client share domain-level authority, CMS templates, brand awareness, and technical SEO health. A naive random split would allow the model to memorize client-level domain performance, causing severe optimistic validation leakage.
* **Design**: We reserve 25% of unique clients strictly for the holdout test set. All URLs belonging to test clients are evaluated together, mirroring how FlyRank deploys to new or distinct web properties.
* **Temporal Discipline**: All features are computed strictly from `month=2026-03` signals, and the binary target is computed strictly from the outcome window (`month=2026-04`).

## setup

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 1. Connect DuckDB and configure Hugging Face token
con = duckdb.connect()

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if hf_token:
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
    print("DuckDB Hugging Face secret configured.")
else:
    print("Warning: HF_TOKEN not found.")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{WAREHOUSE}/fact_content_daily_performance"

# 2. Extract March 2026 Features + April 2026 Ground Truth with Client ID
df = con.execute(f"""
    WITH march_obs AS (
        SELECT
            content_hash_id,
            ANY_VALUE(client_hash_id) AS client_hash_id,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
            SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0) AS ctr,
            SUM(sessions_organic) AS organic_sessions,
            SUM(ga4_total_engagement_sec)::FLOAT / NULLIF(SUM(sessions_organic), 0) AS avg_engagement_sec,
            COUNT(DISTINCT report_date) AS active_days
        FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 10 IS TRUE
    ),
    april_outcome AS (
        SELECT
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks
        FROM read_parquet('{FACT_DAILY}/month=2026-04/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.*,
        COALESCE(a.april_clicks, 0) AS april_clicks,
        CASE WHEN COALESCE(a.april_clicks, 0) > m.clicks THEN 1 ELSE 0 END AS target_rebound
    FROM march_obs m
    LEFT JOIN april_outcome a ON m.content_hash_id = a.content_hash_id
""").df().fillna(0)

# Compute Week 4 heuristic baseline score for direct benchmark comparison
def expected_ctr(pos):
    if pos <= 3: return 0.20
    elif pos <= 6: return 0.08
    elif pos <= 10: return 0.03
    elif pos <= 20: return 0.015
    else: return 0.005

df['expected_ctr'] = df['avg_position'].apply(expected_ctr)
df['ctr_gap'] = (df['expected_ctr'] - df['ctr']).clip(lower=0)
df['baseline_score'] = (
    np.log10(df['impressions'] + 1)
    * df['ctr_gap']
    * (1.0 + (10.0 / df['avg_position'].clip(lower=1.0)))
).round(4)

# 3. Client-Grouped Train/Test Split (75% Train clients, 25% Test clients)
features = [
    'clicks',
    'impressions',
    'avg_position',
    'ctr',
    'organic_sessions',
    'avg_engagement_sec',
    'active_days',
    'ctr_gap'
]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

X_train, y_train = train_df[features], train_df['target_rebound']
X_test, y_test = test_df[features], test_df['target_rebound']

print(f"Total Rows: {len(df):,}")
print(f"Train Rows: {len(train_df):,} across {train_df['client_hash_id'].nunique()} unique clients")
print(f"Test Rows:  {len(test_df):,} across {test_df['client_hash_id'].nunique()} unique clients")
print(f"Target Rebound Rate (Train): {y_train.mean():.3f} | (Test): {y_test.mean():.3f}")

DuckDB Hugging Face secret configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total Rows: 143,206
Train Rows: 110,318 across 33 unique clients
Test Rows:  32,888 across 12 unique clients
Target Rebound Rate (Train): 0.191 | (Test): 0.184


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### 3. Model Training & Comparison Against Week-4 Baseline

We train two models on the 110,318 training instances (33 clients) and benchmark their performance against our Week-4 heuristic baseline score on the 32,888 held-out test instances (12 completely unseen clients).

#### Metrics Evaluated:
1. **ROC-AUC**: Global ranking separation across all decision thresholds.
2. **PR-AUC (Average Precision)**: Performance under the ~18.4% class imbalance.
3. **Precision@100 & Precision@500**: Practical operational capacity metrics. If editorial sprint capacity allows reviewing 100 or 500 URLs, what proportion of top-flagged pages actually achieve traffic recovery?

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Evaluate Week 4 Heuristic Baseline on Test Split
test_baseline_scores = test_df['baseline_score']
auc_baseline = roc_auc_score(y_test, test_baseline_scores)
prauc_baseline = average_precision_score(y_test, test_baseline_scores)

# Precision@K helper function
def precision_at_k(y_true, scores, k=100):
    top_k_indices = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k_indices].mean()

p100_baseline = precision_at_k(y_test, test_baseline_scores, k=100)
p500_baseline = precision_at_k(y_test, test_baseline_scores, k=500)

# 2. Train Reference Logistic Regression (with standard scaling)
log_reg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
log_reg.fit(X_train, y_train)
preds_logreg = log_reg.predict_proba(X_test)[:, 1]

auc_logreg = roc_auc_score(y_test, preds_logreg)
prauc_logreg = average_precision_score(y_test, preds_logreg)
p100_logreg = precision_at_k(y_test, preds_logreg, k=100)
p500_logreg = precision_at_k(y_test, preds_logreg, k=500)

# 3. Train Primary Model: Histogram Gradient Boosted Classifier
hgb = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    random_state=42
)
hgb.fit(X_train, y_train)
preds_hgb = hgb.predict_proba(X_test)[:, 1]

auc_hgb = roc_auc_score(y_test, preds_hgb)
prauc_hgb = average_precision_score(y_test, preds_hgb)
p100_hgb = precision_at_k(y_test, preds_hgb, k=100)
p500_hgb = precision_at_k(y_test, preds_hgb, k=500)

# 4. Compile Model-vs-Baseline Comparison Table
comparison_table = pd.DataFrame({
    'Method': [
        'Week-4 Heuristic Baseline',
        'Logistic Regression (Linear)',
        'HistGradientBoosting (Ensemble)'
    ],
    'ROC-AUC': [auc_baseline, auc_logreg, auc_hgb],
    'PR-AUC': [prauc_baseline, prauc_logreg, prauc_hgb],
    'Precision@100': [p100_baseline, p100_logreg, p100_hgb],
    'Precision@500': [p500_baseline, p500_logreg, p500_hgb]
}).round(4)

print("--- Model vs. Baseline Performance on Unseen Clients (n=32,888) ---")
display(comparison_table)

--- Model vs. Baseline Performance on Unseen Clients (n=32,888) ---


,Method,ROC-AUC,PR-AUC,Precision@100,Precision@500
0,Week-4 Heuristic Baseline,0.6227,0.2390,0.24,0.234
1,Logistic Regression (Linear),0.5955,0.2217,0.21,0.212
2,HistGradientBoosting (Ensemble),0.6729,0.3089,0.83,0.556


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### 4. Feature Importance, Error Analysis & Failure Modes

#### Feature Interpretation:
Permutation feature importance reveals which signals genuinely drive the model's out-of-domain ranking power:
* **Impression Volume & Search Demand**: Strongest drivers of absolute recovery probability.
* **CTR Gap & SERP Position Interactions**: Non-linear splits prevent false-positive inflation on zero-click SERP feature pages that tripped up the simple baseline rule.

#### What Do the Errors Look Like?
1. **False Positives (High Model Confidence, No Rebound)**:
   * **SERP Cannibalization & Google AI Overviews**: Pages ranking position 1–3 with high impressions that never generate clicks because search intent is fulfilled directly inside the Google SERP snippet.
   * **Stale Evergreen Intent Shifts**: Pages targeting topics that experienced sudden algorithmic intent shifts towards video or forum (Reddit) results.
2. **False Negatives (Low Model Confidence, Strong Rebound)**:
   * **Long-Tail Discovery Surges**: Articles that had low impressions in March but caught a seasonal query spike or competitor drop-off in April that was unheralded by historical search volume.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell A
from sklearn.inspection import permutation_importance
# 1. Compute Permutation Feature Importance on Held-Out Test Set
perm_importance = permutation_importance(hgb, X_test, y_test, n_repeats=5, random_state=42, scoring='roc_auc')

feat_imp_df = pd.DataFrame({
    'Feature': features,
    'Importance_Mean': perm_importance.importances_mean,
    'Importance_Std': perm_importance.importances_std
}).sort_values(by='Importance_Mean', ascending=False).reset_index(drop=True)

print("--- Permutation Feature Importance (ROC-AUC Impact) ---")
display(feat_imp_df.round(4))

# 2. Inspect Top False Positives (High Confidence Errors)
test_results = test_df.copy()
test_results['predicted_prob'] = preds_hgb
test_results['prediction_error'] = test_results['predicted_prob'] - test_results['target_rebound']

# Top 5 False Positives (Model predicted high recovery, but URL remained flat/declined)
false_positives = test_results[test_results['target_rebound'] == 0].sort_values(by='predicted_prob', ascending=False).head(5)
print("\n--- Top False Positives (Model Predicted Rebound, Actual: No Rebound) ---")
display(false_positives[['content_hash_id', 'predicted_prob', 'impressions', 'clicks', 'avg_position', 'ctr', 'april_clicks']])

# Top 5 False Negatives (Model predicted low recovery, but URL surged)
false_negatives = test_results[test_results['target_rebound'] == 1].sort_values(by='predicted_prob', ascending=True).head(5)
print("\n--- Top False Negatives (Model Predicted Stagnation, Actual: Strong Rebound) ---")
display(false_negatives[['content_hash_id', 'predicted_prob', 'impressions', 'clicks', 'avg_position', 'ctr', 'april_clicks']])

--- Permutation Feature Importance (ROC-AUC Impact) ---


,Feature,Importance_Mean,Importance_Std
0,impressions,0.1615,0.0019
1,active_days,0.0330,0.0022
2,avg_position,0.0138,0.0003
3,ctr,0.0111,0.0009
4,clicks,0.0049,0.0004
5,ctr_gap,0.0026,0.0002
6,organic_sessions,0.0010,0.0017
7,avg_engagement_sec,-0.0012,0.0002



--- Top False Positives (Model Predicted Rebound, Actual: No Rebound) ---


,content_hash_id,predicted_prob,impressions,clicks,avg_position,ctr,april_clicks
26881,content_b0fb7d22899634c2,0.935881,33.0,1.0,0.484848,0.030303,0.0
34215,content_1db634b4093a38eb,0.917001,50.0,1.0,4.900000,0.020000,1.0
111303,content_13a57ad66e1899f9,0.896184,307.0,5.0,6.576547,0.016287,4.0
31792,content_f4ef8d97a0e1e293,0.865871,819.0,2.0,5.418803,0.002442,0.0
85973,content_3a8feffbe9c50e2f,0.864266,490.0,1.0,7.548980,0.002041,1.0



--- Top False Negatives (Model Predicted Stagnation, Actual: Strong Rebound) ---


,content_hash_id,predicted_prob,impressions,clicks,avg_position,ctr,april_clicks
72816,content_21333ea627c3504b,0.013292,23.0,0.0,73.521739,0.0,1.0
76776,content_423eddfb6e69f1e1,0.014393,56.0,0.0,78.589286,0.0,1.0
24475,content_e5ae0f3c0c64adb0,0.014457,31.0,0.0,79.967742,0.0,1.0
16477,content_d5b712921c05df84,0.014896,43.0,0.0,84.372093,0.0,1.0
123888,content_853e10d7d32a9cb0,0.017552,10.0,0.0,43.600000,0.0,1.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.